# pycograd — compiling the DSL to JAX

This is the **JAX** companion to [`pycograd_demo`](pycograd_demo.ipynb): the same ambient-weights DSL — `with params{...} as weights`, a `$ |> ...` pipeline, and `weights.grad(objective)` — but every net is **trained and run on JAX** instead of pycograd's numpy tape. Two seams do it:

* **`weights.grad(objective, backend="jax", jit=True)`** differentiates the same ambient objective with JAX's own autodiff, and compiles that gradient **once** (the net is traced a single time, then reused every step).
* **`compile_to(forward, "jax")`** runs a forward on JAX's own tensors.

No framework is imported until a backend is named. We open with a linear classifier and an MLP, show forward/gradient parity against the numpy tape, freeze a parameter, then look at how `jit=True` compiles the step on JAX.

**Requires** the `notebook` extra: `pip install pycograd[notebook]`.

In [1]:
%load_ext pycograd

## Setup: losses and data

There is no op library to import — pycograd differentiates raw numpy, so the loss and activations are just numpy used as pipe stages. The models below are all rank-2 matmuls, so they compile cleanly on every backend.

In [2]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow's C++ logging
import warnings; warnings.filterwarnings("ignore")

import numpy as np
from pycograd import compile_to, get_backend, frozen

# --- losses & activations: ordinary numpy, used as pipe stages -----------------
def softmax_ce(logits, onehot):
    z = logits - np.max(logits, axis=1, keepdims=True)          # stabilize
    logp = z - np.log(np.sum(np.exp(z), axis=1, keepdims=True)) # log-softmax
    return -np.mean(np.sum(onehot * logp, axis=1))

def relu(z):     return np.maximum(0.0, z)
def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

# --- data: three Gaussian blobs, 3-way classification --------------------------
rng = np.random.default_rng(0)
m = 40
centers = np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.5, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]

# --- this notebook compiles to JAX ----------------------------------------
BACKEND = "jax"

def train(weights, objective, steps, lr):
    """Train on JAX: every step's gradient comes from JAX's own autodiff, and
    jit=True compiles that gradient ONCE (the net is traced a single time, not per step)."""
    first = last = None
    for _ in range(steps):
        value, grads = weights.grad(objective, backend=BACKEND, jit=True)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, lr)                 # plain in-place SGD on the numpy weights
    return first, last

def accuracy(logits):
    return float(np.mean(np.argmax(np.asarray(logits), axis=1) == labels))

## Example 1 — a linear softmax classifier, trained on JAX

The forward is one pipe stage, `$ @ w + b`; `weights.grad(objective, backend="jax", jit=True)` differentiates it with JAX and compiles the step once. Nothing about the model definition changed from the numpy demo — only where the gradient comes from.

In [3]:
with params{
    w = np.zeros((2, 3))
    b = np.zeros(3)
} as weights:
    forward = $ |> $ @ w + b                       # logits; the loss does the softmax
    objective = |> X |> forward |> softmax_ce($, Y)
    first, last = train(weights, objective, 200, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", accuracy(logits))

loss 1.0986 -> 0.0040
train accuracy: 1.0


## Run the forward on JAX's own tensors

`compile_to(forward, "jax")` turns the **same** DSL `forward` into a callable over JAX tensors (the ambient weights are lifted onto the backend automatically).

In [4]:
with params{
    w = np.zeros((2, 3))
    b = np.zeros(3)
} as weights:
    forward = $ |> $ @ w + b
    be = get_backend(BACKEND)
    run = compile_to(forward, BACKEND)             # a forward on JAX's own tensors
    out = run(be.const(X))                         # pass an JAX tensor in, get one out
    native = np.asarray(be.to_numpy(out))
    ref = forward(X)                               # the same forward in numpy

print("compile_to returned a", type(out).__module__.split(".")[0],
      "tensor of shape", tuple(native.shape))
print("matches numpy:", np.allclose(native, ref))

compile_to returned a jaxlib tensor of shape (120, 3)
matches numpy: True


## Gradients agree with the numpy tape

Calling `weights.grad(objective)` with no backend runs pycograd's own numpy tape; with `backend="jax"` the gradient comes from JAX. They match to tight tolerance — and since the tape is finite-difference-checked, that validates the compile path end-to-end.

In [5]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    v_np, g_np = weights.grad(objective)                    # pycograd's numpy tape
    v_be, g_be = weights.grad(objective, backend=BACKEND)   # JAX's own autodiff
    worst = max(np.max(np.abs(np.asarray(g_be[k]) - np.asarray(g_np[k]))) for k in weights)

print("value  numpy=%.6f  %s=%.6f" % (float(v_np), BACKEND, float(v_be)))
print("max |grad_%s - grad_numpy| over all weights: %.1e" % (BACKEND, worst))

value  numpy=1.514232  jax=1.514232
max |grad_jax - grad_numpy| over all weights: 2.2e-16


## Example 2 — a 2-layer MLP, as one pipeline

`relu` is just a pipe stage. Same DSL, same training call — a deeper net.

In [6]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    first, last = train(weights, objective, 300, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", accuracy(logits))

loss 1.8837 -> 0.0007
train accuracy: 1.0


## A frozen parameter

`frozen[...]` holds a weight fixed: its gradient comes back `None` from JAX's autodiff (same semantics as the numpy tape) and `weights.step` skips it.

In [7]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = frozen[np.zeros(16)]                       # held fixed: gradient comes back None
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    _, g_be = weights.grad(objective, backend=BACKEND)
    first, last = train(weights, objective, 300, 0.5)
    logits = forward(X)

print("frozen b1 gradient:", g_be["b1"])            # None -> excluded from JAX's autodiff
print("loss %.4f -> %.4f  |  b1 stayed all-zero: %s" % (first, last, bool(np.all(weights["b1"].value == 0))))
print("train accuracy:", accuracy(logits))

frozen b1 gradient: None
loss 1.5022 -> 0.0008  |  b1 stayed all-zero: True
train accuracy: 1.0


## How `jit=True` compiles on JAX

`train(...)` above passes `jit=True`, so the gradient is built with `jax.jit(jax.value_and_grad(...))` — XLA compiles the gradient and every step reuses it. The objective is traced a single time no matter how many steps run — verify it by counting how often the Python forward actually executes.

In [8]:
traces = {"n": 0}
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    def objective():
        traces["n"] += 1                            # bumped once per real forward / trace
        return (forward_(X) |> softmax_ce($, Y))
    forward_ = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    first, last = train(weights, objective, 50, 0.5)

print("50 steps -> forward traced %d time(s)" % traces["n"])
print("loss %.4f -> %.4f   train accuracy %.3f"
      % (first, last, accuracy(forward_(X))))

50 steps -> forward traced 1 time(s)
loss 1.2553 -> 0.0050   train accuracy 1.000


## More

Richer models — highway nets, self-attention, a full Transformer encoder block — are written in exactly this DSL in [`pycograd_demo`](pycograd_demo.ipynb), and each trains on a framework the same way: add `backend=`/`jit=` to `weights.grad`. The cross-backend conformance suite checks forward + gradient parity for every installed framework:

```
python -m pytest test/test_compile.py
```